# Adaptive RAG

> Notebook generated from [`examples/rag/02_adaptive_rag.py`](../../examples/rag/02_adaptive_rag.py).

| Data | Value |
|------|-------|
| **Dataset** | NQ + TriviaQA (embedded, 6 query types) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** Classifier prints query type → chosen route (CRAG/HyDE/Fusion/Hybrid) → answer.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Adaptive RAG — Intelligent routing of queries to specialized RAG engines
================================================================================
Architecture: SPEC-RAG-006 / prismal.rag.adaptive

Dataset: Natural Questions (NQ) + TriviaQA
  • NQ: 307,373 Google Search questions about Wikipedia articles.
  • TriviaQA: 95,000 trivia questions with evidence from multiple sources.
  • Reference:
    - https://huggingface.co/datasets/google-research-datasets/natural_questions
    - https://huggingface.co/datasets/mandarjoshi/trivia_qa
  • Why: NQ and TriviaQA cover the 6 query types that AdaptiveRAG
    classifies: simple factual, abstract, ambiguous, technical, conversational,
    and multi-hop. They are the standard benchmark for RAG systems.

Adaptive RAG architecture description:
  Classifies the query and routes to the most appropriate engine:
  - FACTUAL_SIMPLE  → CRAG  (direct factual questions)
  - ABSTRACT        → HyDE  ("why?", "how does it work?" questions)
  - AMBIGUOUS       → RAG-Fusion (short or vague queries)
  - TECHNICAL       → Hybrid Search (technical terms, API, code)
  - CONVERSATIONAL  → CRAG  (references to previous conversation)
  - MULTI_HOP       → CRAG  (compound questions "first X, then Y")

Classification:
  - Default: deterministic regex (no LLM cost)
  - Optional: use_llm_classifier=True for edge cases

Usage:
    uv run python examples/rag/02_adaptive_rag.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio

from langchain_core.documents import Document

from prismal.rag.adaptive import AdaptiveRAGEngine, AdaptiveResult, QueryType
from prismal.rag.crag import CRAGPipeline
from prismal.rag.fusion import RAGFusionEngine
from prismal.rag.hybrid import HybridSearchEngine
from prismal.rag.hyde import HyDERetriever
from prismal.rag.vector_store import ChromaVectorStore

## Dataset: NQ + TriviaQA queries classified by type

In [ ]:
ADAPTIVE_QUERIES = [
    # FACTUAL_SIMPLE: direct facts with a precise answer
    {
        "id": "AQ1",
        "query": "In what year was OpenAI founded?",
        "query_type": QueryType.FACTUAL_SIMPLE,
        "dataset": "NQ",
        "reason": "Direct factual question with a specific date",
    },
    {
        "id": "AQ2",
        "query": "How many parameters does GPT-4 have?",
        "query_type": QueryType.FACTUAL_SIMPLE,
        "dataset": "TriviaQA",
        "reason": "Factual question with a specific number",
    },
    # ABSTRACT: conceptual questions like "why?" or "how?"
    {
        "id": "AQ3",
        "query": "Why are Transformer models more efficient than RNNs?",
        "query_type": QueryType.ABSTRACT,
        "dataset": "NQ",
        "reason": "Explanatory question with 'why', requires reasoning",
    },
    {
        "id": "AQ4",
        "query": "How does the attention mechanism work in modern LLMs?",
        "query_type": QueryType.ABSTRACT,
        "dataset": "NQ",
        "reason": "Mechanism-style 'how does it work' question",
    },
    # AMBIGUOUS: short or vague queries
    {
        "id": "AQ5",
        "query": "RAG",
        "query_type": QueryType.AMBIGUOUS,
        "dataset": "NQ",
        "reason": "Single-word query, very ambiguous",
    },
    {
        "id": "AQ6",
        "query": "best AI tools",
        "query_type": QueryType.AMBIGUOUS,
        "dataset": "TriviaQA",
        "reason": "Short, vague query without context",
    },
    # TECHNICAL: technical terms, code, APIs
    {
        "id": "AQ7",
        "query": "ChromaVectorStore.similarity_search() API documentation",
        "query_type": QueryType.TECHNICAL,
        "dataset": "Technical Docs",
        "reason": "camelCase/snake_case notation and explicit API",
    },
    {
        "id": "AQ8",
        "query": "async def vs sync function performance Python asyncio",
        "query_type": QueryType.TECHNICAL,
        "dataset": "Technical Docs",
        "reason": "Programming technical terms with code syntax",
    },
    # CONVERSATIONAL: references to previous context
    {
        "id": "AQ9",
        "query": "Can you explain more about what you mentioned earlier about BERT?",
        "query_type": QueryType.CONVERSATIONAL,
        "dataset": "NQ",
        "reason": "Explicit reference to previous conversation ('what you mentioned')",
    },
    # MULTI_HOP: requires sequential reasoning
    {
        "id": "AQ10",
        "query": "First tell me who created Python, and then how old that language is",
        "query_type": QueryType.MULTI_HOP,
        "dataset": "NQ",
        "reason": "'First X, then Y' structure indicates multi-hop reasoning",
    },
]

## Base documents for the RAG engines

In [ ]:
BASE_DOCUMENTS = [
    Document(
        page_content=(
            "OpenAI was founded in December 2015 by Sam Altman, Greg Brockman, "
            "Ilya Sutskever, Elon Musk, and others. The company develops AI systems "
            "including the GPT series of language models. GPT-4, released in 2023, "
            "is estimated to have over 1 trillion parameters, though OpenAI has not "
            "officially confirmed the exact number."
        ),
        metadata={"source": "openai_wiki", "chunk_id": "0", "topic": "openai"},
    ),
    Document(
        page_content=(
            "Transformers are more efficient than RNNs primarily because they process "
            "all tokens in parallel rather than sequentially. RNNs must process tokens "
            "one at a time, making them slow for long sequences. Transformers use "
            "self-attention to directly relate any two positions in a sequence, "
            "solving the vanishing gradient problem that plagues RNNs. This allows "
            "Transformers to be trained on much larger datasets and achieve better results."
        ),
        metadata={"source": "transformer_wiki", "chunk_id": "1", "topic": "transformers"},
    ),
    Document(
        page_content=(
            "Retrieval-Augmented Generation (RAG) combines retrieval systems with "
            "generative models. The attention mechanism in modern LLMs uses queries, "
            "keys, and values to compute weighted sums of value vectors. For each token, "
            "the model computes attention scores against all other tokens, then uses "
            "these scores to aggregate information. Multi-head attention runs this "
            "process in parallel with different learned projections."
        ),
        metadata={"source": "rag_attention_wiki", "chunk_id": "2", "topic": "rag_attention"},
    ),
    Document(
        page_content=(
            "ChromaDB is a vector database designed for AI applications. The "
            "ChromaVectorStore class wraps ChromaDB and exposes similarity_search(). "
            "Python async/await syntax allows writing concurrent code that looks like "
            "synchronous code. The asyncio module provides the event loop infrastructure. "
            "BERT uses bidirectional training unlike GPT's unidirectional approach."
        ),
        metadata={"source": "technical_docs", "chunk_id": "3", "topic": "technical"},
    ),
    Document(
        page_content=(
            "Python was created by Guido van Rossum and first released in 1991. "
            "In 2024, Python is 33 years old. It has become the most popular "
            "programming language for data science and machine learning. "
            "The best AI tools in 2024 include LangChain, LlamaIndex, AutoGen, "
            "CrewAI, and the prismal framework."
        ),
        metadata={"source": "python_wiki", "chunk_id": "4", "topic": "python"},
    ),
]


async def setup_adaptive_rag(collection_name: str = "adaptive_rag_example") -> AdaptiveRAGEngine:
    """Set up the Adaptive RAG engine with all sub-engines.

    Args:
        collection_name: Base ChromaDB collection name.

    Returns:
        AdaptiveRAGEngine configured with CRAG, HyDE, RAG-Fusion and Hybrid.
    """
    # Shared vector store
    store = ChromaVectorStore(collection_name=collection_name)
    store.add_documents(BASE_DOCUMENTS)
    print(f"  ✓ {len(BASE_DOCUMENTS)} documents indexed in ChromaDB")

    # Build sub-engines
    crag = CRAGPipeline(vector_store=store)
    hyde = HyDERetriever(vector_store=store)
    fusion = RAGFusionEngine(vector_store=store, n_queries=4)
    hybrid = HybridSearchEngine(vector_store=store, alpha=0.5)

    # Build BM25 index for Hybrid Search
    corpus = [doc.page_content for doc in BASE_DOCUMENTS]
    doc_ids = [doc.metadata["chunk_id"] for doc in BASE_DOCUMENTS]
    hybrid.build_index(corpus=corpus, doc_ids=doc_ids)
    print(f"  ✓ BM25 index built for {len(corpus)} documents")

    # Adaptive engine with all sub-engines available
    return AdaptiveRAGEngine(
        crag_pipeline=crag,
        hyde_retriever=hyde,
        fusion_engine=fusion,
        hybrid_engine=hybrid,
        use_llm_classifier=False,  # regex classifier (no LLM cost)
    )


def print_adaptive_result(query_info: dict, result: AdaptiveResult) -> None:
    """Print the result from the adaptive engine."""
    classification_correct = result.query_type == query_info["query_type"]
    icon = "✓" if classification_correct else "✗"

    print(f"\n[{query_info['id']}] {query_info['query']}")
    print(f"  Dataset       : {query_info['dataset']}")
    print(f"  Expected type : {query_info['query_type'].value}")
    print(
        f"  Detected type : {result.query_type.value}  {icon}  (confidence: {result.confidence:.2f})"
    )
    print(f"  Engine used   : {result.strategy_used}")
    print(f"  Chunks obtained: {len(result.chunks)}")

    for chunk in result.chunks[:2]:  # show at most 2 chunks
        print(f"    • [{chunk.relevance_score:.2f}] {chunk.source}: {chunk.content[:80]}...")
    print("─" * 70)


async def main() -> None:
    print("=" * 70)
    print("  Adaptive RAG — Dataset: NQ + TriviaQA (query classification)")
    print("=" * 70)

    # Setup
    print("\n[RAG sub-engine initialization]")
    engine = await setup_adaptive_rag()
    print("  ✓ Adaptive RAG engine ready with 4 sub-engines")

    # Routing table
    print("\n[Routing table (regex classifier)]")
    routing = [
        ("FACTUAL_SIMPLE", "CRAG", "direct facts, dates, names"),
        ("ABSTRACT       ", "HyDE", "questions 'why?', 'how does it work?'"),
        ("AMBIGUOUS      ", "RAG-Fusion", "short/vague queries (< 3 tokens)"),
        ("TECHNICAL      ", "Hybrid", "snake_case, CamelCase, API, SDK"),
        ("CONVERSATIONAL ", "CRAG", "references to previous context"),
        ("MULTI_HOP      ", "CRAG*", "'first X, then Y' structure"),
    ]
    for qtype, motor, criteria in routing:
        print(f"  {qtype} → {motor:12s} ({criteria})")
    print("  * GraphRAG reserved for Phase B")

    # Evaluate all queries
    print(f"\n[Evaluating {len(ADAPTIVE_QUERIES)} NQ + TriviaQA queries]")

    correct_classifications = 0
    engine_usage: dict[str, int] = {}

    for query_info in ADAPTIVE_QUERIES:
        result = await engine.search(query_info["query"], k=3)
        print_adaptive_result(query_info, result)

        if result.query_type == query_info["query_type"]:
            correct_classifications += 1

        engine_usage[result.strategy_used] = engine_usage.get(result.strategy_used, 0) + 1

    # Statistical summary
    print("\n[Statistical summary]")
    accuracy = correct_classifications / len(ADAPTIVE_QUERIES)
    print(
        f"  Correct classification : {correct_classifications}/{len(ADAPTIVE_QUERIES)} ({accuracy:.0%})"
    )

    print("\n  Distribution of engines used:")
    for engine_name, count in sorted(engine_usage.items(), key=lambda x: -x[1]):
        bar = "█" * count
        print(f"    {engine_name:15s}: {bar} ({count})")

    # Demonstrate LLM classifier
    print("\n[LLM vs Regex Classifier]")
    print("  Regex (default):")
    print("    + No additional LLM cost")
    print("    + Deterministic and reproducible")
    print("    - May fail on ambiguous edge cases")
    print("  LLM (use_llm_classifier=True):")
    print("    + Better precision on edge cases")
    print("    + Understands natural language nuances")
    print("    - Extra cost of an LLM call")
    print("    - Falls back to regex if the LLM fails")


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()